# Semana del 8: caso random con sampler y buffer grande

Es **la idea del profe**: los datos se juntan con acciones al azar en vez de con el actor, y se mide si el actor que se entrena soñando con esos datos aprende a conseguir galletas.

**Por qué ahora:** en los dos casos base de esta semana (`caso_base_decoder` y `caso_base_decoder_sampler`, 300.000 pasos) el agente consiguió solo 2 o 3 galletas en 143 partidas, menos que un agente al azar (~7). Además, apenas veía una galleta, el actor se volvía casi determinista y dejaba de explorar. El cuello de botella es la exploración, que es lo que ataca el azar.

Respecto de `caso_base_decoder_sampler` cambian dos cosas:

1. `CASO = "random"`: las acciones al juntar datos son al azar. El actor igual se entrena (soñando) y es el que se evalúa.
2. `BUFFER = 300_000`: el buffer guarda todo el run. Con 100.000, en el caso base la primera galleta se borró cerca del paso 135.000 y el sampler se quedó sin nada que mostrar.

Lo demás es igual: decoder, `REWARD_SLICES = 2`, 300.000 pasos, `cookie_fullfixed`, `TRAIN_RATIO = 128`, semilla 0.

Lo que se mira es `episode/eval_score` (10 episodios con el actor aprendido, sin azar, cada 10.000 pasos). `episode/score` aquí es el de un agente al azar y no dice nada del actor.

Con `BUFFER = 300_000` el replay en Drive pesa ~6,6 GB (el triple que en los casos base), así que cada guardado del checkpoint tarda un poco más.

Si la sesión se corta, vuelve a correr las celdas 1 a 6 y después la 9: el run se retoma solo desde el último checkpoint en Drive.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (L4).

## 1. Revisar la GPU que tocó

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Montar Drive

Aquí van los checkpoints y las métricas, para que sobrevivan al corte de sesión y para poder comparar los dos casos al final.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Clonar el repo (o actualizarlo si ya está)

In [ ]:
%%bash
if [ -d /content/r2dreamer-cookie ]; then
  cd /content/r2dreamer-cookie && git pull --ff-only
else
  git clone https://github.com/FabriRandon/r2dreamer-cookie.git /content/r2dreamer-cookie
fi

## 4. Instalar

r2dreamer necesita Python 3.11 y Colab suele traer una versión más nueva, así que se crea un entorno aparte con `uv`. Toma unos minutos y hay que repetirlo en cada sesión nueva.

In [ ]:
%%bash
pip install -q uv
cd /content/r2dreamer-cookie
uv venv --python 3.11 .venv
uv pip install --python .venv/bin/python -e ".[cookie]" 2>&1 | tail -3

## 5. Comprobar que todo quedó bien

In [ ]:
%%bash
cd /content/r2dreamer-cookie
.venv/bin/python - <<'EOF'
import torch, gymnasium, cookie_env
from envs.cookie import Cookie
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU")
a = torch.randn(256, 256, device="cuda", dtype=torch.float16)
print("float16 ok:", bool(torch.isfinite((a @ a).sum())))
env = Cookie("partial")
print("galleta ok:", env.reset()["image"].shape, "| acciones:", env.action_space.n, "(deberían ser 3)")
EOF

## 6. Configurar el run

Respecto de `caso_base_decoder_sampler.ipynb` cambian `CASO` (allá `"base"`), `BUFFER` (allá 100.000) y `LOGDIR`.

- `BUFFER`: cuántas transiciones guarda el buffer. Con 300.000 no se borra nada en todo el run.
- `REWARD_SLICES`: cuántas de las 16 secuencias de cada batch se cortan alrededor de una galleta.
- `APAGAR_AL_TERMINAR`: con `True`, la sesión se apaga sola cuando termina el entrenamiento y deja de gastar unidades. Todo queda guardado en Drive; las celdas 10 y 11 se corren después, en una sesión nueva (celdas 2 y 6, y luego la que quieras). Ponlo en `False` si vas a estar mirando y quieres seguir con la 10 y la 11 en la misma sesión.

In [ ]:
CASO = "random"  # la idea del profe: datos juntados al azar

TAREA = "cookie_fullfixed"
PASOS = 300_000  # antes 100_000
TRAIN_RATIO = 128
SEMILLA = 0
REP_LOSS = "dreamer"  # antes "r2dreamer": con decoder
REWARD_SLICES = 2
BUFFER = 300_000  # en los casos base era 100_000

# Apaga la sesión cuando termina el entrenamiento (celda 9), para poder dejarlo corriendo sin gastar de más.
APAGAR_AL_TERMINAR = True

LOGDIR = f"/content/drive/MyDrive/galleta/semana8/{TAREA}_s{SEMILLA}_{CASO}_decoder_sampler_buffer300k"

OPCIONES = (
    f"env=cookie env.task={TAREA} env.train_ratio={TRAIN_RATIO} "
    f"seed={SEMILLA} model.rep_loss={REP_LOSS} buffer.reward_slices={REWARD_SLICES} "
    f"trainer.random_policy={CASO == 'random'}"
)
print(LOGDIR)
print(OPCIONES)

## 7. Prueba corta (unos 10 minutos)

Antes del run largo: corre 1500 pasos en disco local (no en Drive), con un buffer chico que no se guarda y un solo episodio de evaluación. Comprueba que todo funcione y estima cuánto va a tardar el run completo.

In [ ]:
!rm -rf /content/prueba_semana8_random
!cd /content/r2dreamer-cookie && timeout 1200 .venv/bin/python -u train.py {OPCIONES} \
    env.steps=1500 env.eval_episode_num=1 trainer.update_log_every=500 trainer.save_every=1e9 \
    buffer.max_size=1e4 trainer.save_buffer=False \
    logdir=/content/prueba_semana8_random > /content/prueba_semana8_random.log 2>&1; \
    tail -n 5 /content/prueba_semana8_random.log

In [ ]:
import json

lineas = [json.loads(l) for l in open("/content/prueba_semana8_random/metrics.jsonl")]
fps = [l["fps/fps"] for l in lineas if l.get("fps/fps", 0) > 0]
evaluo = any("episode/eval_score" in l for l in lineas)
print("evaluación ok:", evaluo)
if not fps:
    print("No alcanzó a medir la velocidad. Revisa el final de /content/prueba_semana8_random.log (arriba).")
else:
    v = sum(fps) / len(fps)
    print(f"{v:.1f} pasos por segundo → {PASOS:,} pasos en unas {PASOS / v / 3600:.1f} horas, "
          "más unos minutos por cada evaluación")

## 8. Abrir TensorBoard

Ábrelo **antes** de entrenar: mientras la celda de entrenamiento corre, Colab no deja ejecutar otras, pero TensorBoard se sigue actualizando (botón de recargar arriba a la derecha).

Lo que importa está en la pestaña Scalars:

- `episode/eval_score`: galletas por episodio del actor aprendido. Es lo que dice si funcionó.
- `train/rew`: si el modelo predice recompensa. Con el sampler debería subir después de la primera galleta y, con el buffer grande, ya no volver a 0.
- `episode/score`: galletas del agente al azar mientras junta datos. Debería rondar 0,05 por episodio; sirve para contar cuántas galletas tiene el buffer.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/galleta/semana8"

## 9. Entrenar

Arriba aparece una barra de progreso con el tiempo que falta, y debajo va corriendo el log:

```
random:  26%|██▌       | 26181/100000 [04:00<45:05, 25.9paso/s, evaluando]
```

Se lee así: pasos hechos / total, `[tiempo transcurrido<tiempo que falta, velocidad]`. El tiempo que falta se calcula con la velocidad promedio de todo el run, así que se va afinando a medida que avanza. Mientras evalúa, la barra dice `evaluando` y se queda quieta unos minutos.

La barra aparece recién con la primera línea del log que trae un paso: antes compila el modelo y hace la primera evaluación, y eso toma unos minutos.

En el log quedan una línea por episodio (`episode/score`), una cada 2500 pasos con las pérdidas y una por evaluación (`episode/eval_score`).

Para parar antes, usa el botón de detener de la celda: el último checkpoint queda en Drive y basta con volver a correrla para seguir. Si la detienes así, la sesión no se apaga sola.

Para dejarlo corriendo de noche: deja la pestaña abierta y que el PC no entre en suspensión (la pantalla sí se puede apagar). Con `APAGAR_AL_TERMINAR = True`, al terminar se apaga sola.

In [ ]:
import re
import subprocess

from tqdm.auto import tqdm

comando = (
    f".venv/bin/python -u train.py {OPCIONES} env.steps={PASOS} trainer.update_log_every=2500 "
    f'trainer.save_every=1e4 buffer.max_size={BUFFER} logdir="{LOGDIR}"'
)
proceso = subprocess.Popen(comando, shell=True, cwd="/content/r2dreamer-cookie", text=True, bufsize=1,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
barra = None
detenido = False
try:
    for linea in proceso.stdout:
        paso = re.match(r"\[(\d+)\]", linea)
        if paso:
            paso = min(int(paso.group(1)), PASOS)
            if barra is None:
                # Parte desde el paso en que va, por si el run se retomó a mitad de camino.
                barra = tqdm(total=PASOS, initial=paso, unit="paso", smoothing=0, desc=CASO)
            barra.update(paso - barra.n)
            if "eval_score" in linea:
                barra.set_postfix_str("")
            if " progress " in linea:
                continue  # la barra ya muestra el avance
        if barra is not None and linea.startswith("Evaluating"):
            barra.set_postfix_str("evaluando")
        print(linea, end="")
    proceso.wait()
except KeyboardInterrupt:
    detenido = True
    proceso.terminate()
    proceso.wait()
    print("Detenido. El último checkpoint quedó en Drive; para seguir, vuelve a correr esta celda.")
finally:
    if barra is not None:
        barra.close()
if proceso.returncode not in (0, None, -15):
    print(f"El entrenamiento terminó con un error (código {proceso.returncode}). Revisa las últimas líneas de arriba.")

if APAGAR_AL_TERMINAR and not detenido:
    from google.colab import drive, runtime

    # Drive sube los archivos con atraso: esto espera a que todo quede guardado antes de apagar.
    print("Guardando todo en Drive y apagando la sesión para que deje de gastar unidades...")
    drive.flush_and_unmount()
    runtime.unassign()

## 10. Comparar con los casos base

Grafica el `eval_score` de este run junto a los dos casos base de esta semana. La línea punteada es un agente al azar, como referencia: si una curva no la supera, ese caso no aprendió nada.

In [ ]:
import json, pathlib
import matplotlib.pyplot as plt

AZAR = 0.05  # galletas por episodio de un agente al azar con 3 acciones (simulación de 200 episodios)

RUNS = {
    "base con decoder": "/content/drive/MyDrive/galleta/semana8/cookie_fullfixed_s0_base_decoder",
    "base con decoder + sampler": "/content/drive/MyDrive/galleta/semana8/cookie_fullfixed_s0_base_decoder_sampler",
    "random con decoder + sampler, buffer 300k (este)": LOGDIR,
}

fig, ax = plt.subplots(figsize=(7, 4))
for nombre, carpeta in RUNS.items():
    archivo = pathlib.Path(carpeta) / "metrics.jsonl"
    if not archivo.exists():
        print(f"Todavía no hay métricas de '{nombre}' en {carpeta}")
        continue
    puntos = [json.loads(l) for l in open(archivo)]
    puntos = [(p["step"], p["episode/eval_score"]) for p in puntos if "episode/eval_score" in p]
    if puntos:
        pasos, puntaje = zip(*puntos)
        ax.plot(pasos, puntaje, marker="o", label=nombre)
        print(f"{nombre}: última evaluación en el paso {pasos[-1]:,} → {puntaje[-1]:.2f} galletas por episodio")
ax.axhline(AZAR, color="gray", linestyle="--", label="agente al azar")
ax.set_xlabel("pasos del ambiente")
ax.set_ylabel("galletas por episodio (evaluación)")
ax.set_title(f"{TAREA}, semilla {SEMILLA}")
ax.legend()
plt.show()

## 11. Ver al agente jugar

Carga el agente entrenado desde Drive (`latest.pt`) y lo pone a jugar un episodio completo con lo que aprendió, igual que en la evaluación. El video queda guardado en la carpeta del run, así que se puede volver a ver cualquier otro día. Para correrla en una sesión nueva, antes hay que correr las celdas 1 a 6 (no hace falta volver a entrenar).

- A la izquierda está el mapa. En las variantes parciales, lo iluminado es lo que alcanza a ver el agente.
- A la derecha está la imagen que recibe el agente en ese paso.
- Arriba van el paso, las galletas que lleva y la jugada que eligió.

Con `SEMILLA_EPISODIO` cambian las esquinas donde aparece la galleta, para ver al agente en otras situaciones. Tarda uno o dos minutos.

In [ ]:
SEMILLA_EPISODIO = 0
VIDEO = f"{LOGDIR}/agente_episodio{SEMILLA_EPISODIO}.mp4"

!cd /content/r2dreamer-cookie && .venv/bin/python watch.py --logdir "{LOGDIR}" --out "{VIDEO}" \
    --seed {SEMILLA_EPISODIO} 2>&1 | tail -n 1

from IPython.display import Video
Video(VIDEO, embed=True, width=760)